# Week 2 - Day 3: Evaluator-Optimizer (reflection)

> **How this notebook works.** Each **TODO** cell is real code with a few blanks: `___`. Replace every `___` (the comment on that line tells you what goes there), run the cell, then run the **check** below it and compare what it prints with the `# expected:` comments. After each part there is **Classwork**: a new problem that you build **yourself**, from an empty cell. Use the code above as your guide, but type every line yourself; open the hint only if you are stuck. Finished early? Try the *If you finish early* idea under it.

A good writer reads the first draft again, finds the problems and fixes them. A workflow can do the same:
- one model call **writes** a draft (the **optimizer**),
- another call **grades** it with a fixed rule (the **evaluator**), and gives **feedback**,
- if the grade is `fail`, the graph goes **back** to the writer with the feedback. This is a **loop**.

```
START --> write_draft --> grade_draft --"Accepted"--> END
               ^                |
               +---"Rejected"---+   (with the feedback)
```

This is the **evaluator-optimizer** pattern from the LangGraph guide **[Workflows and agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents)**. We add one thing the docs leave out: a **limit**, so the loop cannot run forever.

Today needs **no web search**: we use notes saved from a real search, so everybody works on the same text.

## Setup
Today you need **one** free key, the same as Week 1:
- **Groq** (console.groq.com -> API Keys), saved as `GROQ_API_KEY`

**On Google Colab:** click the key icon (Secrets) on the left, add the key, and switch **Notebook access** on. **On your computer:** put it in a `.env` file.

> **Tips.** The free Groq plan allows about 8,000 tokens a minute. If a cell pauses, it is waiting for the limit. If a cell ends with `RateLimitError`, wait a minute and run it again. If a cell ends with `NameError: name '___' is not defined`, a blank is still empty. If a check ends with another `NameError`, run the TODO cell above it first.

In [ ]:
# What: installs the libraries for this notebook (about 1 minute, once per Colab session).
# Why:  Colab starts without LangChain, LangGraph or the Groq and Tavily connectors.
# How:  %pip is the notebook command for pip: -q keeps it quiet, -U gets the newest versions.
%pip install -qU langchain langgraph langchain-groq langchain-tavily python-dotenv

In [ ]:
# What: puts your API keys into the environment, where LangChain looks for them.
# Why:  keys are secrets: they never go into the notebook text, so the notebook can be shared safely.
# How:  on Colab, userdata reads the Secrets panel; on your computer, load_dotenv reads a .env file.
import os
from dotenv import load_dotenv

load_dotenv()                                                   # on your computer: reads the keys from a .env file
try:
    from google.colab import userdata                           # on Google Colab: reads the keys from Secrets
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    pass

print("Groq key found:", "GROQ_API_KEY" in os.environ)

In [ ]:
# What: the chat model for the whole week: gpt-oss-120b, hosted by Groq.
# Why:  one model object, created once, is used by every step below.
# How:  init_chat_model("provider:model") picks the right connector. temperature=0 gives the most repeatable answers.
#       max_retries=10: when the free plan's speed limit is reached, it waits and tries again instead of failing.
from langchain.chat_models import init_chat_model

llm = init_chat_model("groq:openai/gpt-oss-120b", temperature=0, max_retries=10)

from typing import Literal                          # Literal: only the listed words are allowed
from typing_extensions import TypedDict             # TypedDict: describes the keys of a state
from pydantic import BaseModel, Field               # BaseModel + Field: the classes for structured output
from langgraph.graph import StateGraph, START, END  # the graph builder, and its start and end points
from langchain_core.callbacks import get_usage_metadata_callback   # the token meter
from IPython.display import Image, display

In [ ]:
# Given: the question and the saved notes (from a real web search, with their links).
# What: the question, the notes that every draft must use, and two drafts to test the evaluator.
# Why:  saved notes = everybody grades the same text, and no search tokens are spent.
# How:  triple quotes (""" ... """) make one text that spans many lines.
QUESTION = "Is it worth learning Rust in 2026?"

NOTES = """- Career demand for Rust programmers in 2026 is high, especially for systems roles that pay 15-25% more than application Rust positions. (https://www.kore1.com/hire-rust-developers-2026)
- Rust offers memory safety without a garbage collector. In benchmarks it is up to 12 times faster than Go in memory-heavy tests. (https://curohq.com/blogs/exploring-robust-rust-alternatives-for-system-programming)
- The median salary for Rust developers in the UK is 100,000 pounds a year. There are 37 permanent Rust jobs in the UK, up from 7 in 2025. (https://www.itjobswatch.co.uk/jobs/uk/rust%20developer.do)"""

# Two drafts to test the evaluator. The bad one invents numbers that are not in the notes.
BAD_DRAFT = "Rust is used by 90% of all developers and every Rust job pays 300,000 dollars, so everyone should learn it now."
GOOD_DRAFT = ("Rust looks worth learning for systems work: demand is high and systems roles pay 15-25% more "
              "(https://www.kore1.com/hire-rust-developers-2026). It is memory safe without a garbage collector "
              "(https://curohq.com/blogs/exploring-robust-rust-alternatives-for-system-programming).")

---
# Part 1 - The evaluator

The evaluator is a model with **structured output**, like the router of Day 1. It fills two fields:
- `grade`: only `"pass"` or `"fail"`,
- `feedback`: what to fix.

The **rule** is in the `description` of `grade`: a draft passes only if **every fact is in the notes** and every fact has its link. A clear, checkable rule is the most important part of this pattern.

In [ ]:
# TODO 1: the feedback class and the evaluator. Fill the 2 blanks.
# What: the evaluator: a model that must fill a grade ("pass" or "fail") and feedback.
# Why:  the loop needs a fixed word to decide, and feedback that tells the writer what to fix.
# How:  the rule lives in the description of grade: the model reads it and checks the draft against it.
class Feedback(BaseModel):
    grade: Literal["pass", ___] = Field(description="pass only if every fact in the draft is in the notes, and every fact has its link")   # the second word: "fail"
    feedback: str = Field(description="If the draft fails, say exactly what to fix")

evaluator = llm.with_structured_output(___, method="json_schema", strict=True)     # the class the model must fill

In [ ]:
# CHECK 1: test the evaluator before you trust it.
# How it works: the evaluator gets the notes AND the draft: it cannot check a fact without the source.
bad = evaluator.invoke(f"Grade this draft.\nNotes:\n{NOTES}\n\nDraft:\n{BAD_DRAFT}")
print("Bad draft :", bad.grade, "-", bad.feedback)       # expected: fail, and it names the invented numbers

good = evaluator.invoke(f"Grade this draft.\nNotes:\n{NOTES}\n\nDraft:\n{GOOD_DRAFT}")
print("Good draft:", good.grade, "-", good.feedback)     # expected: pass

## Classwork 1 · Try to fool the evaluator (about 10 minutes)
**Build it yourself** in the empty cell below. A loop is only as good as its evaluator, so attack yours.

**Steps**
1. Write a draft (a text) where a true fact from the notes has **no link**.
2. Write a draft where a number is **changed a little** (for example 15-25% becomes 20-30%), with its link.
3. Grade both drafts with `evaluator`, the same way as CHECK 1, and print the grade and the feedback of each.

A strict evaluator fails both. If one passes, **you fooled it**: make the rule in `Feedback` stricter (for example *"the numbers are exactly the same as in the notes, and every sentence ends with its link"*), run TODO 1 again, then run your cell again.

<details><summary><b>Stuck? Open a hint</b></summary>

Each draft is a text in quotes. To grade one: `evaluator.invoke(f"Grade this draft.\nNotes:\n{NOTES}\n\nDraft:\n{my_draft}")`, then print `.grade` and `.feedback`. A `for` loop over your two drafts saves typing.
</details>

In [ ]:
# CLASSWORK 1: two tricky drafts, both graded by evaluator.

# Write your code here.

**If you finish early:** write a draft that is correct and has its links, but is very rude. Does it pass? Should it? What would you add to the rule?

---
# Part 2 - The loop

The state keeps the draft, the grade, the feedback, and a counter: `attempts`.
- The **writer** reads the feedback (if there is any) and adds 1 to `attempts`.
- The **gate** after the evaluator has **two exits**: the grade is `pass`, or the writer has tried **3 times**. The limit stops a loop that would never end and would spend all your tokens.

`state.get("feedback", "")` reads a key that may not exist yet: on the first round there is no feedback, so it gives `""` instead of an error.

In [ ]:
# Given: the state.
# What: everything the loop must remember between rounds.
# How:  draft, grade and feedback change every round; attempts counts the rounds.
class State(TypedDict):
    question: str
    notes: str
    draft: str
    grade: str
    feedback: str
    attempts: int

In [ ]:
# TODO 2: the writer and the evaluator nodes. Fill the 2 blanks.
# What: the writer node and the evaluator node.
# Why:  the writer must see the feedback from the last round, and count its attempts.
# How:  state.get("feedback", "") gives "" on the first round (no feedback yet) instead of an error.
#       Every run of write_draft adds 1 to attempts.
def write_draft(state: State):
    prompt = f"Answer in at most 100 words: {state['question']}\nUse only these notes, and put the link after each fact:\n{state['notes']}"
    if state.get("feedback", ""):                                # there is feedback from the evaluator
        prompt = prompt + f"\nFix this problem from your last draft: {state['feedback']}"
    msg = llm.invoke(prompt)
    return {"draft": msg.content, "attempts": state.get("attempts", 0) + ___}     # one more attempt

def grade_draft(state: State):
    result = evaluator.invoke(f"Grade this draft.\nNotes:\n{state['notes']}\n\nDraft:\n{state['draft']}")
    return {"grade": result.grade, "feedback": result.___}               # the second field of Feedback

In [ ]:
# TODO 3: the gate after the evaluator. Two exits: a pass, or the limit. Fill the blank.
# What: the gate after the evaluator, with two exits.
# Why:  without a limit, an evaluator that is never happy would loop forever and spend all your tokens.
# How:  stop if the grade is "pass" OR the writer has tried 3 times.
def route_draft(state: State):
    if state["grade"] == "pass" or state["attempts"] >= ___:      # the limit: 3 attempts
        return "Accepted"
    return "Rejected + Feedback"

In [ ]:
# TODO 4: build the graph. The gate can go BACK to the writer: that is the loop. Fill the blank.
# What: the loop graph: write -> grade -> END, or back to write.
# How:  "Rejected + Feedback" points back to write_draft: an edge that goes back is a loop.
builder = StateGraph(State)
builder.add_node("write_draft", write_draft)
builder.add_node("grade_draft", grade_draft)
builder.add_edge(START, "write_draft")
builder.add_edge("write_draft", "grade_draft")
builder.add_conditional_edges("grade_draft", route_draft, {"Accepted": END, "Rejected + Feedback": ___})   # back to the writer
optimizer_workflow = builder.compile()

In [ ]:
# Draw it: look for the arrow that goes back to write_draft.
display(Image(optimizer_workflow.get_graph().draw_mermaid_png()))

In [ ]:
# CHECK 2: run the loop on the saved notes, and count the tokens.
# How it works: the loop ends with a pass (or at the limit); the meter counts every round.
with get_usage_metadata_callback() as cb:
    state = optimizer_workflow.invoke({"question": QUESTION, "notes": NOTES})

print(state["draft"])
print()
print("Attempts:", state["attempts"])     # expected: 1, 2 or 3
print("Grade   :", state["grade"])        # expected: pass (or fail after 3 attempts: then read the feedback)
print("Tokens  :", cb.usage_metadata)

---
# Part 3 - Watch the loop fix a bad draft

To see the loop **repair** something, we grade the bad draft first and start the graph with its **feedback**, so the writer must fix it. `attempts` starts at 1, because one draft already exists. `stream` prints each node as it runs.

In [ ]:
# What: grade the bad draft, then start the loop with its feedback, so the writer must fix it.
# Why:  to watch a real repair (a good first draft would stop after one round).
# How:  stream(..., stream_mode="updates") gives one update per node as it runs.
#       attempts starts at 1 because one draft already exists.
feedback = evaluator.invoke(f"Grade this draft.\nNotes:\n{NOTES}\n\nDraft:\n{BAD_DRAFT}").feedback
print("Feedback on the bad draft:", feedback)
print()

for step in optimizer_workflow.stream({"question": QUESTION, "notes": NOTES, "feedback": feedback, "attempts": 1}, stream_mode="updates"):
    print(step)          # one line for each node that runs
    print()

**Talk about it:** when is a writer + evaluator loop worth its extra tokens, and when is it not? Think of a quick chat answer and of facts a customer will read.

## Classwork 2 · Your own loop: an email improver (about 30 minutes)
**Build it yourself** in the two empty cells below: a writer + evaluator loop for a new job, a short email to a teacher asking for a deadline extension. Use Part 1 and Part 2 as your guide, but type every line yourself.

- **Your rule:** the email is polite, has at most 80 words, and its last sentence is a question.
- **The limit:** 3 attempts.

```
START --> write_email --> grade_email --"Accepted"--> END
               ^                |
               +---"Rejected"---+
```

**Steps**
1. **2a:** a class `EmailGrade` with `grade` (`"pass"` or `"fail"`, with your rule in the description) and `feedback`. Make `email_grader`, then run its check.
2. **2b:** a state `EmailState` with `request`, `email`, `grade`, `feedback` and `attempts`.
3. A writer node `write_email` (reads the feedback if there is any, adds 1 to `attempts`), and a grader node `grade_email`.
4. A gate `route_email` with two exits: a pass, or 3 attempts.
5. The graph with the loop. Compile it as `email_loop`, then run the last check.

<details><summary><b>Stuck? Open a hint</b></summary>

- 2a is like `Feedback` in TODO 1.
- The nodes and the gate are like TODO 2 and TODO 3: `state.get("feedback", "")` and `state.get("attempts", 0) + 1`.
- The graph is like TODO 4: `"Rejected + Feedback"` goes back to `"write_email"`.
</details>

In [ ]:
# CLASSWORK 2a: the class EmailGrade and the model email_grader.

# Write your code here.

In [ ]:
# CHECK 2a: test the grader first.
print(email_grader.invoke("Grade this email:\nGive me more time for the homework. I was busy.").grade)     # expected: fail (not polite, no question)
print(email_grader.invoke("Grade this email:\nDear Ms. Sharma, I was sick this week and could not finish the Python homework. "
                          "Could I please have two more days to hand it in?").grade)                       # expected: pass

In [ ]:
# CLASSWORK 2b: EmailState, write_email, grade_email, route_email, and the graph email_loop.

# Write your code here.

In [ ]:
# CHECK 2b: run your loop and count the tokens.
with get_usage_metadata_callback() as cb:
    state = email_loop.invoke({"request": "to my teacher, asking for 2 more days for the Python homework because I was sick"})

print(state["email"])
print()
print("Attempts:", state["attempts"], "| grade:", state["grade"])     # expected: pass, after 1 to 3 attempts
print("Tokens  :", cb.usage_metadata)

**If you finish early:** make your rule stricter (at most 40 words), run 2a and the last check again, and compare the attempts and the tokens. Then use `email_loop.stream(..., stream_mode="updates")` to watch every round.

---
# Today you learned
- **Evaluator-optimizer**: one call writes, another grades with a clear rule and gives feedback, and the graph loops.
- **A loop** in LangGraph is a conditional edge that goes **back** to an earlier node.
- **A limit** (`attempts`) makes sure the loop ends.
- A good evaluator needs a **clear, checkable rule**: you tested it on a bad and a good draft before you trusted it.

**Next class: the project.** You join routing, workers and the evaluator into one research workflow, protect an agent with **guardrails**, and measure the result against a plain agent.